# VideoClone AI Studio v0.1 — Entrada e análise de vídeo

Esta versão implementa a primeira camada do sistema:
- interface Gradio;
- upload de vídeo e imagem de referência;
- inspeção técnica com FFprobe;
- extração de quadros de amostra;
- preparação de um pacote de entrada para a próxima etapa de geração.

**Ainda não gera o vídeo final.** A integração do Wan Animate/VACE será adicionada depois de validarmos entrada, duração, FPS, enquadramento e referências.

Use vídeos e imagens próprios ou com autorização. O fluxo desta versão é voltado a transformações visuais não explícitas.

## 1. Configurar o Colab

Selecione GPU nas configurações do Colab. Para este primeiro módulo, a GPU ainda não é necessária; ela será usada na etapa de geração.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install gradio==5.44.1 opencv-python-headless pillow

In [ ]:
import os, json, shutil, subprocess, zipfile, uuid
from pathlib import Path
from PIL import Image
import gradio as gr

BASE = Path("/content/VideoCloneAI")
INPUTS = BASE / "inputs"
FRAMES = BASE / "frames"
OUTPUTS = BASE / "outputs"
for p in (INPUTS, FRAMES, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

def run_cmd(args):
    proc = subprocess.run(args, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr[-2500:])
    return proc.stdout

def inspect_video(video_path):
    if not video_path:
        raise gr.Error("Envie um vídeo de referência.")
    result = run_cmd([
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=codec_name,width,height,r_frame_rate,nb_frames,duration",
        "-show_entries", "format=duration,size",
        "-of", "json", str(video_path)
    ])
    data = json.loads(result)
    stream = data["streams"][0]
    duration = float(stream.get("duration") or data.get("format", {}).get("duration") or 0)
    fps_text = stream.get("r_frame_rate", "0/1")
    a, b = fps_text.split("/")
    fps = float(a) / max(float(b), 1)
    frames = stream.get("nb_frames")
    report = {
        "codec": stream.get("codec_name"),
        "width": stream.get("width"),
        "height": stream.get("height"),
        "fps": round(fps, 3),
        "duration_seconds": round(duration, 3),
        "frame_count": int(frames) if frames and frames.isdigit() else None,
        "file_size_mb": round(float(data.get("format", {}).get("size", 0))/1024**2, 2)
    }
    return report

def prepare_project(video_path, character_path, character_description,
                    wardrobe, environment, visual_style, sample_count):
    if not video_path:
        raise gr.Error("Envie um vídeo de referência.")
    if not character_path:
        raise gr.Error("Envie uma imagem de referência da personagem.")
    if not character_description.strip():
        raise gr.Error("Descreva a personagem para orientar a consistência visual.")

    job_id = uuid.uuid4().hex[:10]
    job_dir = INPUTS / job_id
    frames_dir = job_dir / "reference_frames"
    frames_dir.mkdir(parents=True, exist_ok=True)

    video_dst = job_dir / ("reference" + Path(video_path).suffix.lower())
    image_dst = job_dir / ("character" + Path(character_path).suffix.lower())
    shutil.copy2(video_path, video_dst)
    shutil.copy2(character_path, image_dst)

    meta = inspect_video(video_dst)
    n = max(2, min(int(sample_count), 12))
    duration = meta["duration_seconds"]
    if duration <= 0:
        raise gr.Error("Não foi possível identificar a duração do vídeo.")

    timestamps = [duration * i / (n - 1) for i in range(n)]
    frame_paths = []
    for i, ts in enumerate(timestamps):
        out = frames_dir / f"frame_{i:02d}.jpg"
        run_cmd(["ffmpeg", "-y", "-ss", f"{ts:.3f}", "-i", str(video_dst),
                 "-frames:v", "1", "-q:v", "2", str(out)])
        frame_paths.append(str(out))

    project = {
        "job_id": job_id,
        "video": str(video_dst),
        "character_reference": str(image_dst),
        "video_metadata": meta,
        "character_description": character_description,
        "wardrobe": wardrobe,
        "environment": environment,
        "visual_style": visual_style,
        "sample_timestamps_seconds": timestamps,
        "status": "prepared_for_generation",
        "next_stage": "Wan 2.2 Animate workflow integration"
    }
    manifest = job_dir / "project.json"
    manifest.write_text(json.dumps(project, indent=2, ensure_ascii=False), encoding="utf-8")

    # Gera uma folha de contato para inspeção rápida dos quadros.
    imgs = [Image.open(p).convert("RGB") for p in frame_paths]
    thumb_w, thumb_h = 320, 220
    sheet = Image.new("RGB", (thumb_w * 2, thumb_h * ((len(imgs)+1)//2)), "white")
    for idx, im in enumerate(imgs):
        im.thumbnail((thumb_w, thumb_h-24))
        x = (idx % 2) * thumb_w
        y = (idx // 2) * thumb_h
        sheet.paste(im, (x, y+20))
    contact = job_dir / "contact_sheet.jpg"
    sheet.save(contact, quality=90)

    zip_path = OUTPUTS / f"videoclone_input_{job_id}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for f in job_dir.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(job_dir))

    summary = (
        f"Projeto preparado: {job_id}\n"
        f"Vídeo: {meta['width']}x{meta['height']} | {meta['fps']} FPS | "
        f"{meta['duration_seconds']} s | {meta['codec']}\n"
        f"Quadros extraídos: {len(frame_paths)}\n"
        f"Pacote: {zip_path}"
    )
    return summary, str(contact), str(manifest), str(zip_path)

with gr.Blocks(title="VideoClone AI Studio v0.1") as demo:
    gr.Markdown("# VideoClone AI Studio")
    gr.Markdown("**v0.1 — Upload, análise técnica e preparação de referências**")
    with gr.Row():
        video = gr.Video(label="Vídeo de referência", sources=["upload"])
        character = gr.Image(label="Imagem de referência da personagem", type="filepath")
    character_desc = gr.Textbox(
        label="Descrição da personagem",
        placeholder="Descreva características visuais consistentes e não sensíveis..."
    )
    with gr.Row():
        wardrobe = gr.Textbox(label="Roupa desejada", placeholder="Ex.: roupa casual...")
        environment = gr.Textbox(label="Ambiente desejado", placeholder="Ex.: quarto iluminado...")
    style = gr.Textbox(label="Estilo visual", value="Fotorealista, iluminação natural, aparência cinematográfica")
    samples = gr.Slider(2, 12, value=6, step=1, label="Quantidade de quadros para análise")
    prepare = gr.Button("Analisar e preparar projeto", variant="primary")
    report = gr.Textbox(label="Relatório", lines=4)
    contact = gr.Image(label="Folha de contato dos quadros")
    manifest = gr.File(label="Manifesto do projeto (JSON)")
    bundle = gr.File(label="Pacote de entrada (ZIP)")
    prepare.click(
        fn=prepare_project,
        inputs=[video, character, character_desc, wardrobe, environment, style, samples],
        outputs=[report, contact, manifest, bundle]
    )

demo.queue()
demo.launch(server_name="0.0.0.0", server_port=7860, share=False, debug=True)

# Exibe a interface dentro do próprio Colab, sem criar link público do Gradio.
from google.colab import output
output.serve_kernel_port_as_window(7860)

## 2. Como testar

1. Execute as células em ordem.
2. Abra o link público temporário gerado pelo Gradio.
3. Envie um vídeo curto e uma imagem de personagem que você tem autorização para usar.
4. Preencha descrição, roupa, ambiente e estilo.
5. Clique em **Analisar e preparar projeto**.
6. Confira a folha de contato: ela mostra quadros amostrados do vídeo original.
7. Baixe o ZIP e guarde no Drive se quiser preservar a sessão.

## 3. Próxima implementação

Depois de validar este módulo, a próxima versão adicionará:
- detecção de movimento de câmera e pose;
- seleção do modo Animate (Mix/Move);
- carregamento do workflow Wan 2.2 no ComfyUI;
- envio automático dos arquivos para o workflow;
- fila de renderização e exportação MP4.

A integração de VACE será um estágio separado para edição de cenário e regiões selecionadas. A geração de regiões não visíveis é inferência criativa, não reconstrução factual do que a câmera não capturou.